# Classificador Gradient Boosting: predição e explicabilidade da retenção molecular

### Alunas: Maria Emily Nayla Gomes da Silva (24019) e Rafaela Beatriz Silva Luz (24021)

Neste notebook, serão avaliadas diferentes **técnicas de aprendizado de máquina**, combinando abordagens empregadas no artigo de referência [1], que disponibilizou as estruturas moleculares no formato **SMILES** e suas respectivas informações de retenção, com estratégias adicionais voltadas à melhoria do desempenho preditivo.

Será dada atenção especial à **sensibilidade (*recall*) para a classe de moléculas retidas**, uma vez que a identificação correta desses compostos é particularmente relevante para a priorização de potenciais *hits*.

Além da avaliação do desempenho preditivo, serão empregadas **abordagens de explicabilidade** para investigar quais descritores físico-químicos apresentam maior contribuição para as predições dos modelos. Dessa forma, o objetivo não se limita à classificação das moléculas, mas também à exploração de possíveis relações entre suas propriedades físico-químicas e a retenção em bactérias Gram-negativas.

> **Objetivo:** integrar **modelagem preditiva** e **explicabilidade**, utilizando os dados não apenas para classificar moléculas, mas também como suporte à geração de hipóteses científicas sobre as propriedades moleculares associadas à retenção.

In [1]:
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim  


import pickle

import kagglehub
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import shapiq
import statsmodels.api as sm

from mpl_toolkits.axes_grid1 import make_axes_locatable
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

## Dataset de treino

O dataset utilizado neste trabalho contém **46 descritores físico-químicos** das moléculas, além da identificação das amostras destinadas aos conjuntos de **treino e teste**. Essas características foram obtidas por metodologia descrita no modelo interpretável. A variável-alvo (o *target*) é a coluna **`Retention`**, em que:

- **0** → não retenção da molécula em bactérias Gram-negativas;
- **1** → retenção da molécula em bactérias Gram-negativas.

In [2]:
df_modelo = pd.read_csv("all_df_model_descriptors.csv")
df_modelo

,SMILES,Clean SMILES,Retention,Split,cLogP,Topological Polar Surface Area,Labute Solvent-Accesible Surface Area,Polar Surface Fraction,Molecular Weight,Molecular Complexity,...,SlogP VSA LogP -0.20 to 0.00,SlogP VSA LogP 0.00 to 0.10,SlogP VSA LogP 0.10 to 0.15,SlogP VSA LogP 0.15 to 0.20,SlogP VSA LogP 0.20 to 0.25,SlogP VSA LogP 0.25 to 0.30,SlogP VSA LogP 0.30 to 0.40,SlogP VSA LogP 0.40 to 0.50,SlogP VSA LogP 0.50 to 0.60,SlogP VSA LogP greater than 0.60
0,C[n]1c(SCC2OCCN(Cc3ccccc3)C2)nnc1CC(CC1)CS1(=O)=O,Cn1c(CC2CCS(=O)(=O)C2)nnc1SCC1CN(Cc2ccccc2)CCO1,0,Train,1.78540,77.32,176.304909,0.438558,436.160283,917.062459,...,34.587366,5.917906,17.808678,35.488272,0.00000,0.000000,0.0,0.000000,0.000000,11.761885
1,CN(c(ccc(OC)c1)c1C=C1CO)C1=O,COc1ccc2c(c1)cc(CO)c(=O)n2C,0,Train,1.03940,51.46,92.706815,0.555083,219.089543,586.568946,...,13.654554,0.000000,5.563451,29.060005,0.00000,10.902925,0.0,0.000000,5.749512,0.000000
2,Nc1cc(C(OCCC(CCO2)C2=O)=O)cnc1Cl,Nc1cc(C(=O)OCCC2CCOC2=O)cnc1Cl,0,Train,1.42720,91.51,114.393106,0.799961,284.056385,506.502792,...,14.268263,5.917906,23.199632,12.263211,5.15311,0.000000,0.0,5.687386,0.000000,11.600940
3,CCCCS(c1nc(CCCC2)c2cc1C#N)(=O)=O,CCCCS(=O)(=O)c1nc2c(cc1C#N)CCCC2,0,Train,2.40588,70.82,113.947677,0.621513,278.108899,615.068515,...,22.678896,11.331113,49.427855,11.092327,0.00000,0.000000,0.0,0.000000,0.000000,0.000000
4,CCNc(ccc(S(C(F)F)(=O)=O)c1)c1N,CCNc1ccc(S(=O)(=O)C(F)F)cc1N,0,Train,1.69690,72.19,92.372452,0.781510,250.058755,474.397371,...,9.837253,0.000000,6.923737,23.094585,0.00000,0.000000,0.0,20.155603,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6401,CN(C)c1nc(N)nc(COc(cc2)cc3c2[o]c2c3cccc2)n1,CN(C)c1nc(N)nc(COc2ccc3oc4ccccc4c3c2)n1,1,Train,2.99820,90.30,143.613196,0.628772,335.138225,1057.897221,...,6.606882,0.000000,5.824404,46.881720,0.00000,21.938489,0.0,11.896679,5.749512,0.000000
6402,CCOc1nc(N2CC(C)CC(C)C2)cnc1,CCOc1cncc(N2CC(C)CC(C)C2)n1,1,Test,2.35760,38.25,103.022726,0.371277,235.168462,359.437137,...,0.000000,11.835812,27.192033,12.393687,0.00000,0.000000,0.0,5.817863,5.879988,0.000000
6403,Cc1ccc(c(C)cc(SCC(N)=O)n2)c2c1C,Cc1ccc2c(C)cc(SCC(N)=O)nc2c1C,1,Train,2.73746,55.98,110.781949,0.505317,260.098334,622.710368,...,4.794537,20.771212,16.690354,23.225061,0.00000,10.902925,0.0,0.000000,0.000000,11.761885
6404,Cc(cc1cc2C#N)c(C)cc1nc2NCCNC(C(F)(F)F)=O,Cc1cc2cc(C#N)c(NCCNC(=O)C(F)(F)F)nc2cc1C,1,Train,2.81372,77.81,136.037743,0.571974,336.119796,824.653047,...,4.794537,25.178587,16.690354,18.199101,0.00000,10.902925,0.0,18.989108,0.000000,0.000000


,SMILES,Clean SMILES,Retention,Split,cLogP,Topological Polar Surface Area,Labute Solvent-Accesible Surface Area,Polar Surface Fraction,Molecular Weight,Molecular Complexity,...,SlogP VSA LogP -0.20 to 0.00,SlogP VSA LogP 0.00 to 0.10,SlogP VSA LogP 0.10 to 0.15,SlogP VSA LogP 0.15 to 0.20,SlogP VSA LogP 0.20 to 0.25,SlogP VSA LogP 0.25 to 0.30,SlogP VSA LogP 0.30 to 0.40,SlogP VSA LogP 0.40 to 0.50,SlogP VSA LogP 0.50 to 0.60,SlogP VSA LogP greater than 0.60
0,C[n]1c(SCC2OCCN(Cc3ccccc3)C2)nnc1CC(CC1)CS1(=O)=O,Cn1c(CC2CCS(=O)(=O)C2)nnc1SCC1CN(Cc2ccccc2)CCO1,0,Train,1.78540,77.32,176.304909,0.438558,436.160283,917.062459,...,34.587366,5.917906,17.808678,35.488272,0.00000,0.000000,0.0,0.000000,0.000000,11.761885
1,CN(c(ccc(OC)c1)c1C=C1CO)C1=O,COc1ccc2c(c1)cc(CO)c(=O)n2C,0,Train,1.03940,51.46,92.706815,0.555083,219.089543,586.568946,...,13.654554,0.000000,5.563451,29.060005,0.00000,10.902925,0.0,0.000000,5.749512,0.000000
2,Nc1cc(C(OCCC(CCO2)C2=O)=O)cnc1Cl,Nc1cc(C(=O)OCCC2CCOC2=O)cnc1Cl,0,Train,1.42720,91.51,114.393106,0.799961,284.056385,506.502792,...,14.268263,5.917906,23.199632,12.263211,5.15311,0.000000,0.0,5.687386,0.000000,11.600940
3,CCCCS(c1nc(CCCC2)c2cc1C#N)(=O)=O,CCCCS(=O)(=O)c1nc2c(cc1C#N)CCCC2,0,Train,2.40588,70.82,113.947677,0.621513,278.108899,615.068515,...,22.678896,11.331113,49.427855,11.092327,0.00000,0.000000,0.0,0.000000,0.000000,0.000000
4,CCNc(ccc(S(C(F)F)(=O)=O)c1)c1N,CCNc1ccc(S(=O)(=O)C(F)F)cc1N,0,Train,1.69690,72.19,92.372452,0.781510,250.058755,474.397371,...,9.837253,0.000000,6.923737,23.094585,0.00000,0.000000,0.0,20.155603,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6401,CN(C)c1nc(N)nc(COc(cc2)cc3c2[o]c2c3cccc2)n1,CN(C)c1nc(N)nc(COc2ccc3oc4ccccc4c3c2)n1,1,Train,2.99820,90.30,143.613196,0.628772,335.138225,1057.897221,...,6.606882,0.000000,5.824404,46.881720,0.00000,21.938489,0.0,11.896679,5.749512,0.000000
6402,CCOc1nc(N2CC(C)CC(C)C2)cnc1,CCOc1cncc(N2CC(C)CC(C)C2)n1,1,Test,2.35760,38.25,103.022726,0.371277,235.168462,359.437137,...,0.000000,11.835812,27.192033,12.393687,0.00000,0.000000,0.0,5.817863,5.879988,0.000000
6403,Cc1ccc(c(C)cc(SCC(N)=O)n2)c2c1C,Cc1ccc2c(C)cc(SCC(N)=O)nc2c1C,1,Train,2.73746,55.98,110.781949,0.505317,260.098334,622.710368,...,4.794537,20.771212,16.690354,23.225061,0.00000,10.902925,0.0,0.000000,0.000000,11.761885
6404,Cc(cc1cc2C#N)c(C)cc1nc2NCCNC(C(F)(F)F)=O,Cc1cc2cc(C#N)c(NCCNC(=O)C(F)(F)F)nc2cc1C,1,Train,2.81372,77.81,136.037743,0.571974,336.119796,824.653047,...,4.794537,25.178587,16.690354,18.199101,0.00000,10.902925,0.0,18.989108,0.000000,0.000000


### Seleção dos atributos (*features*)

As *features* utilizadas nesta etapa foram selecionadas a partir de uma análise prévia de seleção de atributos, realizada no notebook de interpretabilidade. Foram mantidos os descritores físico-químicos identificados como mais relevantes para a modelagem da retenção molecular.

Neste notebook, esses atributos serão explorados por meio de **métodos de explicabilidade**, buscando compreender como cada descritor contribui para as predições do modelo. Essa abordagem permite não apenas auxiliar na priorização de potenciais *leads*, mas também investigar quais características físico-químicas apresentam maior associação com a predição de retenção em bactérias Gram-negativas.

In [3]:
## SEPARANDO OS DADOS EM DOIS DATAFRAMES: TREINO E TESTE

FEATURES = [
    "cLogP", "Topological Polar Surface Area", "Polar Surface Fraction", "Molecular Weight", "Molecular Complexity",
    "Molecular Flexibility", "Rotatable Bonds", "Shape Index", "Fraction C sp3", "Symmetric atoms",
    "Aromatic Atoms", "Carbon Count", "Oxygen Count", "Phosphorus Count", "Sulfur Count",
    "Chlorine Count", "Bromine Count", "Iodine Count", "Stereo Centers", "H-Donors",
    "H-Acceptors", "Aromatic Rings", "Aliphatic Rings", "Saturated Carbocycles", "Aromatic Heterocycles",
    "Saturated Heterocycles", "Aliphatic Heterocycles", "Small Rings", "Rings Closures", "Macrocycles",
    "Spiro Atoms", "Primary Amines", "Secondary Amines", "Tertiary Amines", "Pyridine Rings",
    "H-Pyrrole Nitrogens", "Anilines", "Aliphatic Carboxylic Acids", "Aromatic Carboxylic Acids", "Phenols",
    "Amides", "Formal Charge", "Negative Charge Atoms"
]
TARGET = ['Retention']

SEMENTE_ALEATORIA = 200404

df_treino = df_modelo[df_modelo['Split'] == "Train"]
df_teste = df_modelo[df_modelo['Split'] == "Test"]

X_treino = df_treino.reindex(FEATURES, axis=1).values
y_treino = df_treino.reindex(TARGET, axis=1).values
y_treino = y_treino.ravel()

X_teste = df_teste.reindex(FEATURES, axis=1).values
y_teste = df_teste.reindex(TARGET, axis=1).values
y_teste = y_teste.ravel()

É possível observar que aproximadamente 34% das moléculas pertencem à classe de moléculas retidas (`Retention = 1`), correspondendo a 2.208 amostras. Embora seja observado um desbalanceamento entre as classes, a classe minoritária ainda apresenta um número considerável de exemplos disponíveis para o treinamento.

Nesse contexto, considerando que o modelo utilizará diferentes descritores físico-químicos para caracterizar as moléculas, o conjunto de dados fornece uma base relevante para investigar a capacidade dos modelos de aprendizado de máquina em distinguir moléculas retidas e não retidas.

In [4]:
contagem = df_modelo["Retention"].value_counts()
porcentagem = df_modelo["Retention"].value_counts(normalize=True) * 100

print("Número de amostras por classe:")
print(contagem)

print("\nPorcentagem por classe:")
print(porcentagem.round(2))

Número de amostras por classe:
Retention
0    4198
1    2208
Name: count, dtype: int64

Porcentagem por classe:
Retention
0    65.53
1    34.47
Name: proportion, dtype: float64
Número de amostras por classe:
Retention
0    4198
1    2208
Name: count, dtype: int64

Porcentagem por classe:
Retention
0    65.53
1    34.47
Name: proportion, dtype: float64


## Otimização dos hiperparâmetros

Nesta etapa, será utilizado o **Optuna** para buscar uma combinação adequada de hiperparâmetros para o modelo **`HistGradientBoostingClassifier`**. Esse classificador foi escolhido como uma alternativa baseada em *gradient boosting* com maior eficiência computacional, permitindo acelerar o processo de treinamento e, consequentemente, a exploração de diferentes combinações de hiperparâmetros durante a otimização.

A otimização será realizada em conjunto com validação cruzada (*cross-validation*), utilizando o **recall** como métrica (*scoring*) durante a seleção dos hiperparâmetros. Essa escolha está alinhada ao objetivo deste trabalho de aumentar a capacidade do modelo de identificar corretamente as moléculas pertencentes à classe **`Retention = 1`**.

Dessa forma, a otimização priorizará modelos com maior sensibilidade para moléculas retidas, buscando reduzir a ocorrência de falsos negativos e, consequentemente, favorecer a identificação de potenciais *hits*.

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier

def cria_instancia_modelo(trial):

    parametros = {

        "max_iter": trial.suggest_int(
            "num_iteracoes", 700, 1200
        ),

        "learning_rate": trial.suggest_float(
            "taxa_aprendizado",0.02,0.1,log=True
        ),

        "max_depth": trial.suggest_int(
            "profundidade_maxima",4,8
        ),

        "max_leaf_nodes": trial.suggest_int(
            "max_folhas",40,63
        ),

        "min_samples_leaf": trial.suggest_int(
            "min_exemplos_folha",5,25
        ),

        "l2_regularization": trial.suggest_float(
            "regularizacao_l2",1e-4,5e-2,log=True
        ),

        "max_bins": trial.suggest_int(
            "max_bins",100,200
        ),

        "random_state": SEMENTE_ALEATORIA,
    }

    threshold = trial.suggest_float(
        "threshold", 0.30, 0.50
    )

    modelo = HistGradientBoostingClassifier(
        **parametros
    )

    return modelo, threshold

### Estratégia para seleção do modelo

Para auxiliar na seleção do modelo mais adequado aos objetivos deste trabalho, serão utilizadas duas estratégias: a otimização do *threshold* de classificação e a aplicação de uma penalização para combinações de hiperparâmetros que resultem em acurácia inferior a 70%.

O objetivo é priorizar o aumento da sensibilidade (*recall*) para a classe `Retention = 1`, favorecendo a identificação de moléculas potencialmente retidas. Entretanto, esse aumento não deve ocorrer em detrimento de uma redução da precisão (*precision*), uma vez que isso aumentaria o número de **falsos positivos**.

Nesse contexto, falsos positivos representam moléculas classificadas pelo modelo como retidas, mas que pertencem à classe de não retenção. Um número elevado desses casos poderia levar à priorização de moléculas pouco promissoras em etapas posteriores da busca por potenciais *leads*.

In [6]:
import numpy as np

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, accuracy_score


def funcao_objetivo(trial, X, y, num_folds):

    X = np.asarray(X)
    y = np.asarray(y)

    modelo, threshold = cria_instancia_modelo(trial)

    cv = StratifiedKFold(
        n_splits=num_folds,
        shuffle=True,
        random_state=SEMENTE_ALEATORIA
    )

    recalls = []
    accuracies = []

    for train_idx, val_idx in cv.split(X, y):

        X_train_cv = X[train_idx]
        X_val_cv = X[val_idx]

        y_train_cv = y[train_idx]
        y_val_cv = y[val_idx]

        modelo_fold = clone(modelo)

        modelo_fold.fit(
            X_train_cv,
            y_train_cv
        )

        # Probabilidade da classe Retention = 1
        y_prob = modelo_fold.predict_proba(
            X_val_cv
        )[:, 1]

        # Threshold sendo otimizado pelo Optuna
        y_pred = (
            y_prob >= threshold
        ).astype(int)

        recalls.append(
            recall_score(y_val_cv, y_pred)
        )

        accuracies.append(
            accuracy_score(y_val_cv, y_pred)
        )

    recall_medio = np.mean(recalls)
    accuracy_media = np.mean(accuracies)

    # Penaliza trials com accuracy abaixo de 0.70
    if accuracy_media < 0.70:
        raise optuna.TrialPruned()

    # Entre os modelos com accuracy >= 0.70,
    # maximiza recall
    return recall_medio

In [7]:
import optuna
from optuna import create_study

NOME_DO_ESTUDO = "HGBC-threshold-recall-LASSO_optuna"

# objeto_de_estudo = create_study(
#     direction="maximize",
#     study_name=NOME_DO_ESTUDO,
#     storage=f"sqlite:///{NOME_DO_ESTUDO}.db",
#     load_if_exists=True,
# )

objeto_de_estudo = optuna.load_study(
    study_name=NOME_DO_ESTUDO,
    storage=f"sqlite:///{NOME_DO_ESTUDO}.db"
)

print("Melhor trial:", objeto_de_estudo.best_trial.number)
print("Melhor score:", objeto_de_estudo.best_value)
print("Melhores parâmetros:")
print(objeto_de_estudo.best_params)

Melhor trial: 13
Melhor score: 0.7373737373737373
Melhores parâmetros:
{'num_iteracoes': 867, 'taxa_aprendizado': 0.05742864110811035, 'profundidade_maxima': 5, 'max_folhas': 59, 'min_exemplos_folha': 18, 'regularizacao_l2': 0.0001350767197517648, 'max_bins': 150, 'threshold': 0.3054120331304777}
Melhor trial: 13
Melhor score: 0.7373737373737373
Melhores parâmetros:
{'num_iteracoes': 867, 'taxa_aprendizado': 0.05742864110811035, 'profundidade_maxima': 5, 'max_folhas': 59, 'min_exemplos_folha': 18, 'regularizacao_l2': 0.0001350767197517648, 'max_bins': 150, 'threshold': 0.3054120331304777}


A validação cruzada (*cross-validation*) foi realizada utilizando 30 *folds*. Durante o processo de otimização, o **Optuna** avaliou 20 combinações de hiperparâmetros para o modelo, buscando identificar a configuração que melhor atendesse aos critérios de desempenho estabelecidos.

In [8]:
NUM_FOLDS = 30

def funcao_objetivo_parcial(trial):
    return funcao_objetivo(trial, X_treino, y_treino, NUM_FOLDS)

In [9]:
NUM_TENTATIVAS = 20

# objeto_de_estudo.optimize(funcao_objetivo_parcial, n_trials=NUM_TENTATIVAS)

## Desempenho do modelo

A predição da retenção molecular representa um problema desafiador, especialmente devido à complexidade das características físico-químicas envolvidas. Após a otimização dos hiperparâmetros e do *threshold* de classificação, foi obtido um *threshold* de 0,3054, buscando favorecer a identificação de moléculas pertencentes à classe `Retention = 1` sem comprometer excessivamente o desempenho nas demais métricas.

As métricas obtidas para o modelo foram:

- **Threshold otimizado:** 0,3054
- **Acurácia:** 0,7317
- **Sensibilidade (Recall):** 0,6974
- **Especificidade:** 0,7506
- **Precisão (Precision):** 0,6069
- **ROC-AUC:** 0,8077

Considerando o objetivo deste trabalho, o modelo foi capaz de identificar corretamente aproximadamente 69,7% das moléculas retidas, enquanto classificou corretamente aproximadamente 75,1% das moléculas não retidas. A acurácia global obtida foi de aproximadamente 73,2%, indicando um equilíbrio razoável entre a identificação das duas classes.

A precisão de 0,6069 indica que aproximadamente 60,7% das moléculas classificadas como retidas pertencem efetivamente à classe `Retention = 1` no conjunto de teste. Dessa forma, embora ainda exista uma proporção relevante de falsos positivos, o modelo consegue priorizar moléculas potencialmente retidas mantendo um compromisso entre sensibilidade e precisão.

Além disso, o modelo apresentou ROC-AUC de 0,8077, indicando capacidade de discriminação entre moléculas retidas e não retidas considerando diferentes limiares de classificação.

Como referência, o modelo reportado no trabalho original [1], a partir do qual foram obtidas as informações de **SMILES** e retenção molecular, apresentou:

- **Acurácia:** 0,7627
- **Sensibilidade:** 0,7783
- **Especificidade:** 0,7477
- **Precisão:** 0,7466
- **F1-score:** 0,7621

Em comparação com o modelo de referência, o modelo desenvolvido neste trabalho apresentou menor acurácia, sensibilidade e precisão, porém alcançou uma especificidade ligeiramente superior. A sensibilidade de 0,6974, embora inferior aos 0,7783 reportados na literatura, demonstra que uma parcela considerável das moléculas retidas pode ser recuperada pelo modelo.

In [10]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    roc_auc_score
)

melhor_trial = objeto_de_estudo.best_trial

print(f"Número do melhor trial: {melhor_trial.number}")
print(f"Parâmetros do melhor trial: {melhor_trial.params}")


# Retorna modelo e threshold
modelo, threshold = cria_instancia_modelo(melhor_trial)

modelo.fit(X_treino, y_treino)


# Valores verdadeiros
y_verdadeiro = y_teste

# Probabilidade da classe positiva
y_prob = modelo.predict_proba(X_teste)[:, 1]

# Predição usando o threshold otimizado pelo Optuna
y_previsao = (y_prob >= threshold).astype(int)


# Métricas
acc = accuracy_score(
    y_verdadeiro, y_previsao
)

sens = recall_score(
    y_verdadeiro,y_previsao
)

esp = recall_score(
    y_verdadeiro,y_previsao,pos_label=0
)

prec = precision_score(
    y_verdadeiro,y_previsao
)

roc_auc = roc_auc_score(
    y_verdadeiro,y_prob
)


print(f"""
Threshold otimizado:            {threshold:.4f}
Acurácia do modelo:             {acc:.4f}
Sensibilidade:                  {sens:.4f}
Especificidade:                 {esp:.4f}
Precisão:                       {prec:.4f}
ROC AUC:                        {roc_auc:.4f}
""")

Número do melhor trial: 13
Parâmetros do melhor trial: {'num_iteracoes': 867, 'taxa_aprendizado': 0.05742864110811035, 'profundidade_maxima': 5, 'max_folhas': 59, 'min_exemplos_folha': 18, 'regularizacao_l2': 0.0001350767197517648, 'max_bins': 150, 'threshold': 0.3054120331304777}

Threshold otimizado:            0.3054
Acurácia do modelo:             0.7317
Sensibilidade:                  0.6974
Especificidade:                 0.7506
Precisão:                       0.6069
ROC AUC:                        0.8077

Número do melhor trial: 13
Parâmetros do melhor trial: {'num_iteracoes': 867, 'taxa_aprendizado': 0.05742864110811035, 'profundidade_maxima': 5, 'max_folhas': 59, 'min_exemplos_folha': 18, 'regularizacao_l2': 0.0001350767197517648, 'max_bins': 150, 'threshold': 0.3054120331304777}

Threshold otimizado:            0.3054
Acurácia do modelo:             0.7317
Sensibilidade:                  0.6974
Especificidade:                 0.7506
Precisão:                       0.6069
ROC 

### Calibração das probabilidades do modelo

Após a seleção da melhor combinação de hiperparâmetros pelo Optuna, foi realizada a calibração das probabilidades preditas pelo modelo. Essa etapa é importante neste trabalho, uma vez que a classificação final das moléculas utiliza um *threshold* otimizado, em vez do valor padrão de 0,5.

Além disso, as probabilidades preditas pelo modelo serão utilizadas posteriormente para selecionar e priorizar um subconjunto de moléculas para as próximas etapas do estudo, nas quais será investigada sua interação com o alvo molecular de interesse. Nesse contexto, a calibração busca tornar essas probabilidades mais representativas, contribuindo para uma seleção mais criteriosa dos potenciais *hits* que avançarão para as próximas etapas da investigação. Isto é, busca-se priorizar as moléculas classificadas como positivas que apresentam maior probabilidade predita de pertencerem, de fato, à classe positiva.

A calibração foi realizada utilizando o **`CalibratedClassifierCV`**, com o método **sigmoidal (`method="sigmoid"`)** e validação cruzada. Esse procedimento busca ajustar as probabilidades produzidas pelo classificador para que elas representem de forma mais adequada a confiança associada à classe **`Retention = 1`**.


In [11]:
from sklearn.calibration import CalibratedClassifierCV

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    roc_auc_score
)

# Melhor trial do Optuna
melhor_trial = objeto_de_estudo.best_trial

# Recupera o melhor modelo e o threshold otimizado
base_model, threshold = cria_instancia_modelo(melhor_trial)

# Calibração
modelo = CalibratedClassifierCV(
    estimator=base_model,
    method="sigmoid",
    cv=50
)

# Treina + calibra
modelo.fit(X_treino, y_treino)

# Valores verdadeiros
y_verdadeiro = y_teste

# Probabilidades calibradas
y_prob = modelo.predict_proba(X_teste)[:, 1]

# Aplica o threshold otimizado pelo Optuna
y_previsao = (y_prob >= threshold).astype(int)

# Métricas
acc = accuracy_score(
    y_verdadeiro, y_previsao
)

sens = recall_score(
    y_verdadeiro, y_previsao
)

esp = recall_score(
    y_verdadeiro, y_previsao, pos_label=0
)

prec = precision_score(
    y_verdadeiro, y_previsao
)

roc_auc = roc_auc_score(
    y_verdadeiro, y_prob
)

# Apresentação dos resultados
print(f"""
Número do melhor trial:         {melhor_trial.number}
Threshold otimizado:            {threshold:.4f}

Acurácia do modelo calibrado:    {acc:.4f}
Sensibilidade:                  {sens:.4f}
Especificidade:                 {esp:.4f}
Precisão:                        {prec:.4f}
ROC AUC:                         {roc_auc:.4f}
""")


Número do melhor trial:         13
Threshold otimizado:            0.3054

Acurácia do modelo calibrado:    0.7285
Sensibilidade:                  0.7851
Especificidade:                 0.6973
Precisão:                        0.5888
ROC AUC:                         0.8135


Número do melhor trial:         13
Threshold otimizado:            0.3054

Acurácia do modelo calibrado:    0.7285
Sensibilidade:                  0.7851
Especificidade:                 0.6973
Precisão:                        0.5888
ROC AUC:                         0.8135



### Avaliação da calibração das probabilidades

Para avaliar a qualidade das probabilidades produzidas pelo modelo após a calibração, foi utilizado o **Brier Score**. Essa métrica calcula o erro quadrático médio entre as probabilidades preditas e os valores reais de `Retention`.

O Brier Score varia de 0 a 1, sendo que valores menores indicam probabilidades mais próximas dos resultados observados. Um valor igual a 0 representaria predições probabilísticas perfeitas.

In [12]:
from sklearn.base import clone
from sklearn.metrics import brier_score_loss

# Modelo sem calibração
modelo_nao_calibrado = clone(base_model)

# Treina o modelo
modelo_nao_calibrado.fit(X_treino, y_treino)

# Probabilidades antes da calibração
y_prob_nao_calibrado = modelo_nao_calibrado.predict_proba(X_teste)[:, 1]

# Brier Score antes da calibração
brier_antes = brier_score_loss(
    y_teste,
    y_prob_nao_calibrado
)

# Brier Score depois da calibração
brier_depois = brier_score_loss(
    y_teste,
    y_prob
)

print(f"Brier Score antes da calibração:  {brier_antes:.4f}")
print(f"Brier Score após a calibração:    {brier_depois:.4f}")

Brier Score antes da calibração:  0.1775
Brier Score após a calibração:    0.1674
Brier Score antes da calibração:  0.1775
Brier Score após a calibração:    0.1674


Antes da calibração, o modelo apresentou um **Brier Score de 0,1775**, enquanto, após a calibração, esse valor foi reduzido para **0,1674**. Como valores menores de Brier Score indicam menor discrepância entre as probabilidades preditas e os resultados observados, essa redução evidencia uma melhora na qualidade das estimativas probabilísticas após a calibração.

Embora a diferença entre os valores seja moderada, a redução do Brier Score indica que o processo de calibração contribuiu para tornar as probabilidades preditas mais consistentes com os resultados observados. Dessa forma, as **probabilidades calibradas** serão utilizadas como um dos critérios para a priorização de potenciais *leads* nas etapas subsequentes da investigação.

## Seleção dos potenciais *leads*

Com o modelo treinado, otimizado e calibrado, a próxima etapa consiste em utilizá-lo para selecionar e priorizar potenciais *leads*.

Inicialmente, serão selecionadas as moléculas classificadas pelo modelo como **`Retention = 1`**, de acordo com o *threshold* previamente otimizado. Em seguida, essas moléculas serão ordenadas de acordo com suas probabilidades preditas de retenção, priorizando aquelas para as quais o modelo apresenta maior confiança na classificação positiva.

Busca-se selecionar um subconjunto de 15.000 moléculas com maior potencial de retenção em bactérias Gram-negativas.

In [13]:
df_hit = pd.read_csv("all_df_hit_descriptors.csv")
df_hit.head()

,SMILES,Clean SMILES,cLogP,Topological Polar Surface Area,Labute Solvent-Accesible Surface Area,Polar Surface Fraction,Molecular Weight,Molecular Complexity,Molecular Flexibility,Shape Index,...,SlogP VSA LogP -0.20 to 0.00,SlogP VSA LogP 0.00 to 0.10,SlogP VSA LogP 0.10 to 0.15,SlogP VSA LogP 0.15 to 0.20,SlogP VSA LogP 0.20 to 0.25,SlogP VSA LogP 0.25 to 0.30,SlogP VSA LogP 0.30 to 0.40,SlogP VSA LogP 0.40 to 0.50,SlogP VSA LogP 0.50 to 0.60,SlogP VSA LogP greater than 0.60
0,CC1=CSC(=N1)NC(=O)C=2C=CC=C(C2)S(=O)(=O)N3C=4C...,Cc1csc(NC(=O)c2cccc(S(=O)(=O)N3c4ccccc4CC3C)c2)n1,3.84372,79.37,167.456476,0.473974,413.086783,1154.762424,0.142857,4.681352,...,16.444113,6.923737,28.539105,58.806483,0.0,0.0,0.0,10.818945,0.000000,11.336786
1,COC=1C=C(C)C=C(C1)CNC(=O)N2CCOCC2C,COc1cc(C)cc(CNC(=O)N2CCOCC2C)c1,1.93392,50.80,119.408359,0.425431,278.163043,476.186472,0.150000,4.843807,...,11.281619,6.923737,18.050640,18.199101,0.0,0.0,0.0,4.794537,5.749512,0.000000
2,[O-][N+](=O)C=1C=C(C=CC1N2CCCCC2)C(=O)O,O=C(O)c1ccc(N2CCCCC2)c([N+](=O)[O-])c1,2.28330,83.68,103.983790,0.804741,250.095357,480.965796,0.166667,3.453412,...,0.000000,10.114318,29.620454,18.199101,0.0,0.0,0.0,11.374773,0.000000,0.000000
3,O=C(O)C=1C=CC=CC1SCC(=O)N2CCCC2,O=C(O)c1ccccc1SCC(=O)N1CCCC1,2.09930,57.61,109.980388,0.523821,265.077264,455.304508,0.222222,4.251745,...,4.794537,0.000000,23.199632,29.160952,0.0,0.0,0.0,0.000000,0.000000,11.761885
4,CC=1C=CC(=CC1)S(=O)(=O)N2CCC(CC2)C(=O)O,Cc1ccc(S(=O)(=O)N2CCC(C(=O)O)CC2)cc1,1.48032,74.68,112.456532,0.664079,283.087829,556.810943,0.157895,3.912807,...,14.817828,12.841643,18.405095,29.160952,0.0,0.0,0.0,0.000000,0.000000,0.000000


,SMILES,Clean SMILES,cLogP,Topological Polar Surface Area,Labute Solvent-Accesible Surface Area,Polar Surface Fraction,Molecular Weight,Molecular Complexity,Molecular Flexibility,Shape Index,...,SlogP VSA LogP -0.20 to 0.00,SlogP VSA LogP 0.00 to 0.10,SlogP VSA LogP 0.10 to 0.15,SlogP VSA LogP 0.15 to 0.20,SlogP VSA LogP 0.20 to 0.25,SlogP VSA LogP 0.25 to 0.30,SlogP VSA LogP 0.30 to 0.40,SlogP VSA LogP 0.40 to 0.50,SlogP VSA LogP 0.50 to 0.60,SlogP VSA LogP greater than 0.60
0,CC1=CSC(=N1)NC(=O)C=2C=CC=C(C2)S(=O)(=O)N3C=4C...,Cc1csc(NC(=O)c2cccc(S(=O)(=O)N3c4ccccc4CC3C)c2)n1,3.84372,79.37,167.456476,0.473974,413.086783,1154.762424,0.142857,4.681352,...,16.444113,6.923737,28.539105,58.806483,0.0,0.0,0.0,10.818945,0.000000,11.336786
1,COC=1C=C(C)C=C(C1)CNC(=O)N2CCOCC2C,COc1cc(C)cc(CNC(=O)N2CCOCC2C)c1,1.93392,50.80,119.408359,0.425431,278.163043,476.186472,0.150000,4.843807,...,11.281619,6.923737,18.050640,18.199101,0.0,0.0,0.0,4.794537,5.749512,0.000000
2,[O-][N+](=O)C=1C=C(C=CC1N2CCCCC2)C(=O)O,O=C(O)c1ccc(N2CCCCC2)c([N+](=O)[O-])c1,2.28330,83.68,103.983790,0.804741,250.095357,480.965796,0.166667,3.453412,...,0.000000,10.114318,29.620454,18.199101,0.0,0.0,0.0,11.374773,0.000000,0.000000
3,O=C(O)C=1C=CC=CC1SCC(=O)N2CCCC2,O=C(O)c1ccccc1SCC(=O)N1CCCC1,2.09930,57.61,109.980388,0.523821,265.077264,455.304508,0.222222,4.251745,...,4.794537,0.000000,23.199632,29.160952,0.0,0.0,0.0,0.000000,0.000000,11.761885
4,CC=1C=CC(=CC1)S(=O)(=O)N2CCC(CC2)C(=O)O,Cc1ccc(S(=O)(=O)N2CCC(C(=O)O)CC2)cc1,1.48032,74.68,112.456532,0.664079,283.087829,556.810943,0.157895,3.912807,...,14.817828,12.841643,18.405095,29.160952,0.0,0.0,0.0,0.000000,0.000000,0.000000


In [14]:
## Código para salvar as previsões de Retention = 1 para o conjunto de dados df_top com as maiores probabilidades de serem verdadeiros positivos

# # Mantém df_hit como DataFrame
# X_hit = df_hit.reindex(columns=FEATURES).values

# # # # Probabilidade calibrada de Retention = 1
# df_hit["Retention_Probability"] = model.predict_proba(X_hit)[:, 1]

# # # # Classificação usando o threshold otimizado
# # df_hit["Retention"] = (
# #      df_hit["Retention_Probability"] >= threshold
# # ).astype(int)

# # # Seleciona apenas Retention = 1
# # # e ordena da maior para a menor probabilidade
# # df_15000 = (
# #     df_hit[df_hit["Retention"] == 1]
# #     .sort_values(
# #         by="Retention_Probability",
# #         ascending=False
# #     )
# #     .head(15000)
# #     .copy()
# # )

# # # Salva o resultado
# # df_15000.to_csv(
# #     "top_15000_retention.csv",
# #     index=False
# # )


# print(f"Retention = 1: {(df_hit['Retention'] == 1).sum()}")

In [15]:
## Média de confiança das 15.000 moléculas com maior probabilidade de Retention = 1

df_top = pd.read_csv("top_15000_retention.csv")
df_top["Retention_Probability"].mean()

FileNotFoundError: [Errno 2] No such file or directory: 'top_15000_retention.csv'

FileNotFoundError: [Errno 2] No such file or directory: 'top_15000_retention.csv'

## Como explicar o modelo? Valores de SHAP

Os valores de **SHAP** são utilizados para entender o impacto de cada atributo nas previsões do modelo. Nesse caso, a análise permite identificar quais características moleculares estão associadas à maior ou menor probabilidade de retenção prevista pelo modelo**.

- **SHAP positivo** → o atributo contribui para a previsão de `Retention = 1`;
- **SHAP negativo** → o atributo contribui para a previsão de `Retention = 0`;
- **Maior valor absoluto de SHAP** → maior impacto daquele atributo na previsão.

Assim, além de identificar quais atributos são mais importantes, podemos analisar quais valores desses atributos favorecem ou desfavorecem a retenção prevista pelo modelo.

In [ ]:
X = df_treino[FEATURES]
X

Para este projeto, consideramos relevante compreender os padrões aprendidos pelo modelo durante o treinamento. Dessa forma, os dados de treino serão utilizados para investigar quais características físico-químicas influenciam as predições e como esses atributos contribuem para a classificação das moléculas.

Essa análise busca compreender tanto quais características favorecem a predição de uma molécula como retida (`Retention = 1`) quanto aquelas que contribuem para sua classificação como não retida (`Retention = 0`).

In [ ]:
# Etapa do cálculo do valores de SHAP

# X_100 = shap.utils.sample(X, 100)

# explicador = shap.Explainer(modelo.predict, X_100.values)
# valores_shap = explicador(X.values)

In [ ]:
ARQUIVO = "valores_shap.pkl"

# Aqui para salvar o que acabamos de calcular
# with open(ARQUIVO, "wb") as f:
#     pickle.dump(valores_shap, f)

# Buscando os valores de SHAP salvos no arquivo pickle
with open(ARQUIVO, "rb") as f:
    valores_shap = pickle.load(f)

## Gráfico de enxame de abelhas (*Beeswarm*)

Para obter uma visão global dos valores de **SHAP**, utiliza-se o gráfico de enxame de abelhas (*beeswarm*). Essa visualização permite identificar quais características físico-químicas apresentam maior influência nas predições do modelo e compreender como diferentes valores desses descritores contribuem para a classificação.

A magnitude dos valores de SHAP indica a intensidade da contribuição de cada característica para a predição, enquanto o sinal do valor (valores maiores ou menores que zero) de SHAP indica a direção dessa contribuição. Dessa forma, é possível investigar quais características favorecem a predição de uma molécula como retida (`Retention = 1`) e quais contribuem para sua classificação como não retida (`Retention = 0`).

Além disso, a distribuição das observações no gráfico permite avaliar como valores altos e baixos de cada descritor físico-químico estão associados às diferentes contribuições do modelo, auxiliando na interpretação dos padrões aprendidos durante o treinamento.

In [ ]:
import joblib

# joblib.dump(valores_shap, "valores_shap.pkl")
# joblib.dump(X_100, "X_100_shap.pkl")

FEATURES_PT = [
    "cLogP", "Área de Superfície Polar Topológica", "Fração de Superfície Polar", "Massa Molecular", "Complexidade Molecular",
    "Flexibilidade Molecular", "Ligações Rotacionáveis", "Índice de Forma", "Fração de Carbonos sp³", "Átomos Simétricos",
    "Átomos Aromáticos", "Número de Carbonos", "Número de Oxigênios", "Número de Fósforos", "Número de Enxofres",
    "Número de Cloros", "Número de Bromos", "Número de Iodos", "Centros Estereogênicos", "Doadores de Hidrogênio",
    "Aceitadores de Hidrogênio", "Anéis Aromáticos", "Anéis Alifáticos", "Carbociclos Saturados", "Heterociclos Aromáticos",
    "Heterociclos Saturados", "Heterociclos Alifáticos", "Anéis Pequenos", "Fechamentos de Anel", "Macrociclos",
    "Átomos Espiro", "Aminas Primárias", "Aminas Secundárias", "Aminas Terciárias", "Anéis de Piridina",
    "Nitrogênios Pirrólicos com H", "Anilinas", "Ácidos Carboxílicos Alifáticos", "Ácidos Carboxílicos Aromáticos", "Fenóis",
    "Amidas", "Carga Formal", "Átomos com Carga Negativa"
]

valores_shap = joblib.load("valores_shap.pkl")
X_100 = joblib.load("X_100_shap.pkl")
valores_shap.feature_names = FEATURES_PT

### Interpretação global dos valores de SHAP

O gráfico *beeswarm* evidencia que **cLogP** e **Molecular Complexity** estão entre as características com maior influência nas predições do modelo. Para ambas, valores elevados dos descritores concentram-se predominantemente em regiões de SHAP positivo, indicando contribuição para a predição de retenção molecular. Contudo, os valores baixos dessas características apresentam principalmente contribuições negativas, favorecendo a predição de não retenção.

A **Molecular Flexibility** apresenta um comportamento distinto: valores elevados de flexibilidade concentram-se principalmente na região de SHAP negativo, indicando que moléculas mais flexíveis tendem a contribuir para a predição de não retenção. Adeamis, os valores elevados de **Aliphatic Heterocycles** também estão associados a contribuições negativas para a retenção.

Valores elevados de **Macrocycles** estão associados a valores negativos de SHAP, enquanto valores baixos dessa característica apresentam contribuições próximas de zero ou ligeiramente positivas. Para **Molecular Weight**, valores elevados aparecem predominantemente associados a contribuições negativas, embora exista uma distribuição considerável próxima de SHAP igual a zero.

Esses resultados sugerem que o modelo aprendeu diferentes relações entre os descritores físico-químicos e a retenção molecular. Em particular, maiores valores de cLogP e complexidade molecular favorecem a predição de retenção, enquanto maior flexibilidade molecular, maior presença de heterociclos alifáticos e a presença de macrociclos tendem a direcionar o modelo para a predição de não retenção.

In [ ]:
shap.plots.beeswarm(valores_shap)

O gráfico apresenta a magnitude média da contribuição dos descritores para as predições. A Complexidade Molecular (0,11) e o cLogP (0,10) apresentaram as maiores contribuições, seguidos pela Massa Molecular e Flexibilidade Molecular (0,06). Como são utilizados valores absolutos de SHAP, o gráfico indica a **intensidade da contribuição**.

In [ ]:
shap.plots.bar(valores_shap)

O gráfico mostra que o efeito da **Complexidade Molecular** sobre a predição apresenta um comportamento não linear. Complexidades mais baixas estão mais associadas a valores de SHAP negativos, enquanto o aumento da complexidade, especialmente a partir de aproximadamente 900, tende a contribuir positivamente para a predição de `Retention = 1`.

Em regiões de maior **Complexidade Molecular**, valores mais elevados de cLogP aparecem mais associados a contribuições SHAP positivas. Dessa forma, o modelo parece considerar a combinação entre **maior complexidade molecular e maior cLogP** como favorável à predição de `Retention = 1`.

Entretanto, há sobreposição entre diferentes valores de cLogP ao longo do gráfico, indicando que essa relação não é deve linear. Assim, o efeito da Complexidade Molecular depende, em parte, do contexto fornecido pelo cLogP e por outras características da molécula.

In [ ]:
shap.plots.scatter(valores_shap[:, "Complexidade Molecular"], color=valores_shap[:, "cLogP"])

A seguir, observa-se que as moléculas de maior massa molecular concentram-se principalmente nas regiões de maior complexidade e de SHAP positivo, enquanto menores massas são mais frequentes nas regiões de contribuição negativa.

Esse padrão sugere que o efeito positivo da **Complexidade Molecular** sobre a predição de `Retention = 1` ocorre em conjunto com o aumento da **Massa Molecular**, indicando que essas propriedades são consideradas de forma combinada pelo modelo.

In [ ]:
shap.plots.scatter(valores_shap[:, "Complexidade Molecular"], color=valores_shap[:, "Massa Molecular"])

A presença de **Heterociclos Alifáticos** apresenta uma tendência negativa sobre a predição de retenção. A partir de aproximadamente 1 a 2 heterociclos, os valores de SHAP tornam-se majoritariamente negativos, indicando contribuição para a não retenção.

A coloração também mostra que moléculas com maior número de heterociclos alifáticos tendem a apresentar **maior Massa Molecular**. Entretanto, mesmo nessas condições, a contribuição dos heterociclos permanece negativa e semelhante as anteriores, sugerindo que o efeito dessa característica não muda pelo aumento da massa molecular.

In [ ]:
shap.plots.scatter(valores_shap[:, "Heterociclos Alifáticos"], color=valores_shap[:, "Massa Molecular"])

## Uso do TreeExplainer para o HistGradientBoostingClassifier

Como o `HistGradientBoostingClassifier` é um modelo baseado em **árvores de decisão**, mostra-se interessante explorar métodos de explicabilidade desenvolvidos para essa estrutura. Nesse contexto, o **TreeExplainer** permite investigar a contribuição dos diferentes descritores físico-químicos para as predições realizadas pelo modelo.

In [ ]:
# explicador = shap.TreeExplainer(modelo)
# valores_shap = explicador(X)

# valores_interacao_shap = explicador.shap_interaction_values(X)

In [ ]:
ARQUIVO = "valores_interacao_shap.pkl"

#joblib.dump(valores_interacao_shap, ARQUIVO)

valores_interacao_shap = joblib.load(ARQUIVO)

In [ ]:
interacao = np.abs(valores_interacao_shap).sum(0)

# Zerando as interações de um atributo com ele mesmo
for i in range(interacao.shape[0]):
    interacao[i,i] = 0

# ordenando as colunas por intensidade da interação
indices = np.argsort(-interacao.sum(0))
interacao = interacao[indices,:][:,indices]
colunas = X.columns.tolist()
colunas = [colunas[i] for i in indices]

# Eliminando as interações de um atributo com ele mesmo
for i in range(len(interacao)):
    interacao[i][i] = np.nan

O gráfico apresenta a **força de interação entre os diferentes descritores físico-químicos** nas predições do modelo. As regiões mais claras indicam combinações de atributos com maior contribuição conjunta, enquanto as regiões mais escuras representam interações de menor influência.

Observa-se uma concentração das interações mais relevantes no canto superior esquerdo, principalmente entre descritores como cLogP, Complexidade Molecular, Massa Molecular, Área de Superfície Polar e Flexibilidade Molecular. Esse resultado sugere que o modelo não considera essas propriedades apenas de forma isolada, mas também por meio das relações entre elas durante a predição da retenção molecular.

In [ ]:
fig, axe = plt.subplots(
    ncols=1,
    nrows=1,
    figsize=(10, 10),
    dpi=600,
)

im = axe.imshow(interacao, cmap=plt.get_cmap('cividis'))

# barra de cor
divider = make_axes_locatable(axe)
cax = divider.append_axes("right", size="5%", pad=0.05)
cb = plt.colorbar(im, cax=cax)
cb.set_label("Força da interação entre atributos")
cb.set_ticks([])
cb.set_ticklabels([])

# Configurando os ticks e legenda dos ticks
axe.set_yticks(range(interacao.shape[0]), colunas)
axe.set_xticks(
    range(interacao.shape[0]),
    colunas,
    rotation=50.4,
    horizontalalignment="left",
)
axe.xaxis.tick_top()

### Cálculo das interações entre atributos

Para investigar não apenas a contribuição individual dos descritores, mas também como pares de características atuam conjuntamente nas predições, também foi utilizado o `TabularExplainer` da biblioteca **shapiq**. O índice **k-SII (*k-Shapley Interaction Index*)** foi empregado para quantificar as interações entre os atributos. 

O parâmetro `max_order=2` restringe a análise às contribuições individuais (1ª ordem) e às interações entre pares de atributos (2ª ordem). Assim, é possível investigar, por exemplo, como a combinação entre **cLogP e Complexidade Molecular** contribui para as predições do modelo.

In [ ]:
MAX_INTERACOES = 2

explicador = shapiq.TabularExplainer(
    model=modelo,
    data=X.values,
    max_order=MAX_INTERACOES,
    index="k-SII",
)

In [ ]:
BUDGET = 5000

INDICE_EXEMPLO = 20

valores_shapley = explicador.explain(
    X.values[INDICE_EXEMPLO],
    budget=BUDGET,
    random_state=SEMENTE_ALEATORIA,
)

valores_shapley.feature_names = FEATURES_PT
print(valores_shapley)

O gráfico a seguir apresenta uma explicação local da predição de uma molécula específica, evidenciando as características e interações que contribuíram para deslocar a predição em relação ao valor basal do modelo ($f(x)$).

Para essa molécula, o cLogP (+0,09) apresentou a maior contribuição positiva, enquanto a Complexidade Molecular (-0,08) e a Flexibilidade Molecular (-0,06) contribuíram negativamente. Também são observadas contribuições de interações entre descritores, como Aminas Primárias × Aminas Terciárias (+0,04) e Massa Molecular × Aminas Terciárias (-0,04).

In [ ]:
valores_shapley.plot_waterfall(feature_names=X.columns, abbreviate=False)

O gráfico apresenta as principais contribuições individuais e interações entre descritores para a predição de uma molécula específica. As barras positivas indicam contribuições que aumentam a saída do modelo, enquanto as negativas indicam contribuições na direção oposta.

Aqui, é interessante observar como o gráfico evidencia efeitos conjuntos, como a interação Aminas Primárias × Aminas Terciárias (+0,04) e Massa Molecular × Aminas Terciárias (-0,04).

In [ ]:
valores_shapley.plot_upset(
    n_interactions=10,
    feature_names=X.columns,
    color_matrix=True,
    all_features=False,
)

### Explicação global das interações

Para compreender os padrões gerais aprendidos pelo modelo, foram selecionadas 200 moléculas do conjunto de dados para o cálculo dos valores de interação. Essa abordagem permite avaliar quais características e interações entre descritores apresentam maior contribuição ao longo de diferentes moléculas.

Dessa forma, é possível identificar padrões recorrentes e investigar quais combinações de propriedades físico-químicas são mais relevantes para as predições de retenção do modelo.

In [ ]:
# NUM_EXEMPLOS = 200

# amostra = X.sample(NUM_EXEMPLOS).values

# valores_shapley_global = explicador.explain_X(
#     amostra,
#     budget=BUDGET,
#     n_jobs=-1,
#     random_state=SEMENTE_ALEATORIA,
# )

In [ ]:
## Para salvar os valores shapley global, a amostra e os nomes das features em um arquivo pickle
ARQUIVO = "valores_shapley_global.pkl"

# dados = {
#     "valores_shapley": valores_shapley_global,
#     "amostra": amostra,
#     "FEATURES": FEATURES
# }

# joblib.dump(dados, ARQUIVO)


## Para obter novamente esses dados
dados = joblib.load(ARQUIVO)

valores_shapley_global = dados["valores_shapley"]
amostra = dados["amostra"]
FEATURES = dados["FEATURES"]

Novamente, observa-se que o **cLogP** e **Complexidade Molecular** permanecem entre os descritores com maior impacto nas predições, apresentando contribuições positivas para parte relevante das moléculas analisadas.

Da mesma forma, a **Flexibilidade Molecular** apresenta valores elevados associados a contribuições negativas, padrão também identificado anteriormente no gráfico *beeswarm*. Os **Heterociclos Alifáticos** apresentam comportamento semelhante, com valores elevados associados principalmente a impactos negativos na saída do modelo.

Assim, diferentes abordagens de explicabilidade apresentam padrões consistentes, destacando principalmente **cLogP, Complexidade Molecular e Flexibilidade Molecular** como características relevantes para as decisões do modelo.

In [ ]:
shapiq.beeswarm_plot(
    valores_shapley_global,
    amostra,
    feature_names=X.columns,
    abbreviate=False,
)

## Conclusão

Neste trabalho, foi desenvolvida uma abordagem de aprendizado de máquina e explicabilidade para investigar a retenção de pequenas moléculas em bactérias Gram-negativas, utilizando como ponto de partida os dados experimentais disponibilizados por Le Goff *et al.* [1]. O estudo de referência demonstrou a complexidade desse fenômeno, indicando que nenhuma característica estrutural ou físico-química isolada é suficiente para explicar o comportamento de retenção e mostrando a aplicabilidade de modelos baseados em *gradient boosting* para essa tarefa.

A otimização do modelo, associada ao ajuste do *threshold* e à calibração das probabilidades, permitiu construir uma estratégia voltada à identificação de moléculas potencialmente retidas, considerando a necessidade de controlar falsos positivos. A calibração também possibilitou utilizar as probabilidades preditas como um critério adicional para a priorização de potenciais *leads*.

As análises de explicabilidade realizadas com **SHAP e SHAP-IQ** permitiram investigar os padrões aprendidos pelo modelo. Descritores como cLogP, Complexidade Molecular e Flexibilidade Molecular apresentaram contribuições relevantes e recorrentes nas diferentes análises. Além disso, a investigação das interações mostrou que algumas propriedades físico-químicas podem contribuir para as predições de maneira conjunta, evidenciando a natureza multivariada do problema.

Esses resultados reforçam a importância de utilizar modelos de aprendizado de máquina não apenas como ferramentas de classificação, mas também como instrumentos para explorar relações entre propriedades físico-químicas e retenção molecular. Nesse contexto, a explicabilidade fornece suporte para a geração de hipóteses e para a priorização racional de moléculas que poderão avançar para etapas posteriores de investigação de sua interação com o alvo molecular.

### Referência

[1] Le Goff, F., Hazemann, J., Christen, L. et al. Measurement and prediction of small molecule retention by Gram-negative bacteria based on a large-scale LC/MS screen. Sci Rep 15, 25431 (2025). https://doi.org/10.1038/s41598-025-10208-6